In [ ]:
# import pandas as pd
# import numpy as np
# import os
# from sklearn.cluster import DBSCAN
# from sklearn.preprocessing import StandardScaler
# from datetime import timedelta

# # --- Base Class ---
# class BaseAnalyzer(object):
#     def __init__(self, file_path, output_path, target_sensors):
#         self.file_path = file_path
#         self.output_path = output_path
#         self.target_sensors = target_sensors
#         self.df = None
#         self.df_model = None

#     def load_data(self):
#         if not os.path.exists(self.file_path):
#             raise FileNotFoundError(f"❌ فایل یافت نشد: {self.file_path}")
#         self.df = pd.read_excel(self.file_path)
#         if 'date' in self.df.columns:
#             self.df['date'] = pd.to_datetime(self.df['date'])
#         self.df_model = self.df.dropna(subset=self.target_sensors).copy()

#     def preprocess(self):
#         cols = []
#         for sensor in self.target_sensors:
#             name = f'{sensor}_smooth'
#             self.df_model[name] = self.df_model[sensor].rolling(window=5, center=True).mean()
#             cols.append(name)
#         self.df_model = self.df_model.dropna(subset=cols).copy()
#         return cols

#     def save_output(self, final_df):
#         try:
#             os.makedirs(os.path.dirname(self.output_path), exist_ok=True)
#             final_df.to_excel(self.output_path, index=False)
#             print(f"✅ خروجی در مسیر زیر ذخیره شد:\n{self.output_path}")
#         except Exception as e:
#             print(f"❌ خطا در ذخیره: {e}")

# # --- Derived Class with DBSCAN ---
# class SmartBearingAnalyzer(BaseAnalyzer):
#     def __init__(self, file_path, output_path, target_sensors, eps=0.5, min_samples=5):
#         # مقداردهی مستقیم برای پایداری در نوت‌بوک
#         self.file_path = file_path
#         self.output_path = output_path
#         self.target_sensors = target_sensors
#         self.eps = eps
#         self.min_samples = min_samples
#         self.df = None
#         self.df_model = None

#     def run_analysis(self):
#         self.load_data()
#         smooth_cols = self.preprocess()

#         # ۱. استانداردسازی
#         scaler = StandardScaler()
#         scaled_data = scaler.fit_transform(self.df_model[smooth_cols])

#         # ۲. کلاسترینگ با DBSCAN
#         model = DBSCAN(eps=self.eps, min_samples=self.min_samples)
#         cluster_labels = model.fit_predict(scaled_data)
#         self.df_model['Behavior_Cluster'] = cluster_labels

#         # ۳. محاسبه معیار تخریب (فاصله تا نزدیک‌ترین نقطه هسته در همان خوشه)
#         self.df_model['Degradation_Index'] = self._calculate_degradation_index(scaled_data, cluster_labels, model)

#         # ۴. لیبل‌گذاری
#         self.df_model['Health_Status'] = self.df_model.apply(self._labeling, axis=1)

#         # ۵. فیلتر ۳۰ روز آخر
#         if 'date' in self.df_model.columns:
#             last_dt = self.df_model['date'].max()
#             final_df = self.df_model[self.df_model['date'] >= (last_dt - timedelta(days=30))].copy()
#         else:
#             final_df = self.df_model

#         self.save_output(final_df)

#     def _calculate_degradation_index(self, scaled_data, cluster_labels, model):
#         """
#         محاسبه معیار تخریب:
#         - برای نقاط هسته (core points): فاصله تا مرکز جرم خوشه خودشان
#         - برای نقاط مرزی (border points): فاصله تا نزدیک‌ترین نقطه هسته در همان خوشه
#         - برای نویز (noise, label=-1): حداکثر فاصله مشاهده شده + ۱ (به عنوان بدترین وضعیت)
#         """
#         n_samples = len(scaled_data)
#         degradation = np.zeros(n_samples)
        
#         # پیدا کردن نقاط هسته (core points) برای هر خوشه
#         # در DBSCAN، نقاط هسته نقاطی هستند که حداقل min_samples نقطه در همسایگی eps دارند
#         # ما از core_sample_indices_ استفاده می‌کنیم
        
#         core_mask = np.zeros(n_samples, dtype=bool)
#         core_mask[model.core_sample_indices_] = True
        
#         # محاسبه مرکز جرم برای هر خوشه (فقط از نقاط هسته)
#         unique_clusters = set(cluster_labels) - {-1}  # حذف نویز
#         cluster_centers = {}
        
#         for cluster_id in unique_clusters:
#             cluster_core_points = scaled_data[(cluster_labels == cluster_id) & core_mask]
#             if len(cluster_core_points) > 0:
#                 cluster_centers[cluster_id] = np.mean(cluster_core_points, axis=0)
#             else:
#                 # اگر هیچ نقطه هسته‌ای در خوشه نبود، از میانگین کل نقاط خوشه استفاده کن
#                 cluster_all_points = scaled_data[cluster_labels == cluster_id]
#                 if len(cluster_all_points) > 0:
#                     cluster_centers[cluster_id] = np.mean(cluster_all_points, axis=0)
#                 else:
#                     cluster_centers[cluster_id] = np.zeros(scaled_data.shape[1])
        
#         # محاسبه فاصله برای هر نقطه
#         max_distance = 0
        
#         for i in range(n_samples):
#             cluster_id = cluster_labels[i]
            
#             if cluster_id == -1:  # نویز
#                 # فعلاً صفر بگذار، بعداً مقداردهی می‌کنیم
#                 degradation[i] = 0
#             else:
#                 if i in model.core_sample_indices_:
#                     # نقطه هسته: فاصله تا مرکز جرم خوشه
#                     center = cluster_centers.get(cluster_id, np.zeros(scaled_data.shape[1]))
#                     degradation[i] = np.linalg.norm(scaled_data[i] - center)
#                 else:
#                     # نقطه مرزی: فاصله تا نزدیک‌ترین نقطه هسته در همان خوشه
#                     cluster_core_indices = [idx for idx in model.core_sample_indices_ 
#                                            if cluster_labels[idx] == cluster_id]
                    
#                     if len(cluster_core_indices) > 0:
#                         core_points = scaled_data[cluster_core_indices]
#                         distances_to_cores = np.linalg.norm(core_points - scaled_data[i], axis=1)
#                         degradation[i] = np.min(distances_to_cores)
#                     else:
#                         # اگر هیچ نقطه هسته‌ای نبود، از مرکز جرم استفاده کن
#                         center = cluster_centers.get(cluster_id, np.zeros(scaled_data.shape[1]))
#                         degradation[i] = np.linalg.norm(scaled_data[i] - center)
            
#             if degradation[i] > max_distance:
#                 max_distance = degradation[i]
        
#         # مقداردهی به نقاط نویز (بدترین وضعیت)
#         for i in range(n_samples):
#             if cluster_labels[i] == -1:
#                 degradation[i] = max_distance + 1.0
        
#         return degradation

#     def _labeling(self, row):
#         if row['Behavior_Cluster'] == -1:  # نقاط نویز
#             return "Investigation Needed (Operational Drift)"
#         elif row['Degradation_Index'] > 0.95:
#             return "Investigation Needed (Operational Drift)"
#         elif row['Degradation_Index'] > 0.85:
#             return "Observation Required (Pattern Change)"
#         else:
#             return "Healthy (Optimal Performance)"

# # --- CONFIG & RUN ---
# CONFIG = {
#     "file_path": r'second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx',
#     "output_path": r'outputs\G11\dsas_g11_lubrication_system_clustering\clustering\dsas_g11_lubrication\clustering_g11_lubrication_output2.xlsx',
#     "sensors": ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']
# }

# # اجرا با DBSCAN (پارامترهای eps و min_samples قابل تنظیم هستند)
# analyzer = SmartBearingAnalyzer(
#     CONFIG["file_path"], 
#     CONFIG["output_path"], 
#     CONFIG["sensors"],
#     eps=0.5,      # شعاع همسایگی (قابل تنظیم بر اساس داده)
#     min_samples=5  # حداقل تعداد نقاط برای تشکیل خوشه
# )
# analyzer.run_analysis()

In [ ]:
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime, timedelta
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

# غیرفعال کردن هشدارهای غیرضروری
import warnings
warnings.filterwarnings('ignore')

# --- Base Class ---
class BaseAnalyzer(object):
    def __init__(self, file_path, output_path, target_sensors):
        self.file_path = file_path
        self.output_path = output_path
        self.target_sensors = target_sensors
        self.df = None
        self.df_model = None

    def load_data(self):
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"❌ فایل یافت نشد: {self.file_path}")
        self.df = pd.read_excel(self.file_path)
        if 'date' in self.df.columns:
            self.df['date'] = pd.to_datetime(self.df['date'])
        self.df_model = self.df.dropna(subset=self.target_sensors).copy()
        print(f"✅ داده بارگذاری شد. تعداد رکوردها: {len(self.df_model):,}")

    def preprocess(self):
        cols = []
        for sensor in self.target_sensors:
            name = f'{sensor}_smooth'
            self.df_model[name] = self.df_model[sensor].rolling(window=5, center=True).mean()
            cols.append(name)
        self.df_model = self.df_model.dropna(subset=cols).copy()
        print(f"✅ پیش‌پردازش انجام شد. تعداد رکوردها: {len(self.df_model):,}")
        return cols

    def save_output(self, final_df):
        try:
            os.makedirs(os.path.dirname(self.output_path), exist_ok=True)
            final_df.to_excel(self.output_path, index=False)
            print(f"✅ خروجی در مسیر زیر ذخیره شد:\n{self.output_path}")
        except Exception as e:
            print(f"❌ خطا در ذخیره: {e}")

# --- Derived Class with DBSCAN ---
class SmartBearingAnalyzer(BaseAnalyzer):
    def __init__(self, file_path, output_path, target_sensors, eps=0.5, min_samples=5):
        self.file_path = file_path
        self.output_path = output_path
        self.target_sensors = target_sensors
        self.eps = eps
        self.min_samples = min_samples
        self.df = None
        self.df_model = None

    def run_analysis(self):
        print("="*60)
        print(f"🔄 شروع تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("="*60)
        
        self.load_data()
        smooth_cols = self.preprocess()

        # ۱. استانداردسازی
        print("🔄 مرحله 1: استانداردسازی داده‌ها...")
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(self.df_model[smooth_cols])

        # ۲. کلاسترینگ با DBSCAN
        print(f"🔄 مرحله 2: کلاسترینگ با DBSCAN (eps={self.eps}, min_samples={self.min_samples})...")
        model = DBSCAN(eps=self.eps, min_samples=self.min_samples)
        cluster_labels = model.fit_predict(scaled_data)
        self.df_model['Behavior_Cluster'] = cluster_labels
        
        # آمار خوشه‌ها
        unique_clusters = set(cluster_labels)
        n_clusters = len([x for x in unique_clusters if x != -1])
        n_noise = sum(1 for x in cluster_labels if x == -1)
        print(f"   تعداد خوشه‌ها: {n_clusters}")
        print(f"   تعداد نقاط نویز: {n_noise:,} ({n_noise/len(cluster_labels)*100:.2f}%)")

        # ۳. محاسبه معیار تخریب
        print("🔄 مرحله 3: محاسبه شاخص تخریب...")
        self.df_model['Degradation_Index'] = self._calculate_degradation_index(scaled_data, cluster_labels, model)
        print(f"   محدوده شاخص تخریب: {self.df_model['Degradation_Index'].min():.4f} تا {self.df_model['Degradation_Index'].max():.4f}")

        # ۴. لیبل‌گذاری
        print("🔄 مرحله 4: لیبل‌گذاری وضعیت سلامت...")
        self.df_model['Health_Status'] = self.df_model.apply(self._labeling, axis=1)
        
        # نمایش توزیع وضعیت‌ها
        status_counts = self.df_model['Health_Status'].value_counts()
        print(f"\n📊 توزیع وضعیت‌ها:")
        for status, count in status_counts.items():
            print(f"   {status}: {count:,} ({count/len(self.df_model)*100:.2f}%)")

        # ۵. فیلتر ۳۰ روز آخر
        print("🔄 مرحله 5: فیلتر کردن داده‌های ۳۰ روز آخر...")
        if 'date' in self.df_model.columns:
            last_dt = self.df_model['date'].max()
            start_date = last_dt - timedelta(days=30)
            final_df = self.df_model[self.df_model['date'] >= start_date].copy()
            print(f"   بازه خروجی: {start_date} تا {last_dt}")
            print(f"   تعداد رکوردهای ۳۰ روز آخر: {len(final_df):,}")
        else:
            final_df = self.df_model
            print("   ⚠️ ستون 'date' وجود ندارد، تمام داده‌ها ذخیره می‌شوند.")

        # نمایش آمار نهایی
        print(f"\n📊 آمار نهایی:")
        print(f"   کل رکوردها: {len(self.df_model):,}")
        print(f"   رکوردهای ۳۰ روز آخر: {len(final_df):,}")
        
        # نمایش نمونه‌هایی که نیاز به بررسی دارند
        investigation_needed = final_df[final_df['Health_Status'].str.contains('Investigation Needed')]
        if len(investigation_needed) > 0:
            print(f"\n⚠️ تعداد رکوردهای نیازمند بررسی: {len(investigation_needed):,}")
            print(f"   درصد نیازمند بررسی: {len(investigation_needed)/len(final_df)*100:.2f}%")

        self.save_output(final_df)
        
        print("="*60)
        print(f"✅ تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} کامل شد")
        print("="*60)
        
        return final_df

    def _calculate_degradation_index(self, scaled_data, cluster_labels, model):
        n_samples = len(scaled_data)
        degradation = np.zeros(n_samples)
        
        core_mask = np.zeros(n_samples, dtype=bool)
        core_mask[model.core_sample_indices_] = True
        
        unique_clusters = set(cluster_labels) - {-1}
        cluster_centers = {}
        
        for cluster_id in unique_clusters:
            cluster_core_points = scaled_data[(cluster_labels == cluster_id) & core_mask]
            if len(cluster_core_points) > 0:
                cluster_centers[cluster_id] = np.mean(cluster_core_points, axis=0)
            else:
                cluster_all_points = scaled_data[cluster_labels == cluster_id]
                if len(cluster_all_points) > 0:
                    cluster_centers[cluster_id] = np.mean(cluster_all_points, axis=0)
                else:
                    cluster_centers[cluster_id] = np.zeros(scaled_data.shape[1])
        
        max_distance = 0
        
        for i in range(n_samples):
            cluster_id = cluster_labels[i]
            
            if cluster_id == -1:
                degradation[i] = 0
            else:
                if i in model.core_sample_indices_:
                    center = cluster_centers.get(cluster_id, np.zeros(scaled_data.shape[1]))
                    degradation[i] = np.linalg.norm(scaled_data[i] - center)
                else:
                    cluster_core_indices = [idx for idx in model.core_sample_indices_ 
                                           if cluster_labels[idx] == cluster_id]
                    
                    if len(cluster_core_indices) > 0:
                        core_points = scaled_data[cluster_core_indices]
                        distances_to_cores = np.linalg.norm(core_points - scaled_data[i], axis=1)
                        degradation[i] = np.min(distances_to_cores)
                    else:
                        center = cluster_centers.get(cluster_id, np.zeros(scaled_data.shape[1]))
                        degradation[i] = np.linalg.norm(scaled_data[i] - center)
            
            if degradation[i] > max_distance:
                max_distance = degradation[i]
        
        for i in range(n_samples):
            if cluster_labels[i] == -1:
                degradation[i] = max_distance + 1.0
        
        return degradation

    def _labeling(self, row):
        if row['Behavior_Cluster'] == -1:
            return "Investigation Needed (Operational Drift)"
        elif row['Degradation_Index'] > 0.95:
            return "Investigation Needed (Operational Drift)"
        elif row['Degradation_Index'] > 0.85:
            return "Observation Required (Pattern Change)"
        else:
            return "Healthy (Optimal Performance)"

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز)
    """
    print("="*60)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - تحلیل خوشه‌بندی سیستم روغن‌کاری با DBSCAN")
    print("="*60)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 10:00")
    print("   - ساعت 10:05")
    print("   - ساعت 10:10")
    print("="*60)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*60)
    
    last_run_time = None  # فقط برای جلوگیری از اجرای مجدد در یک زمان
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            # بررسی زمان‌های مشخص
            if current_time in ["22:13", "22:15", "22:17"]:
                # فقط چک می‌کنیم که در همین زمان دوبار اجرا نشود
                if last_run_time != current_time:
                    print("\n" + "="*60)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*60)
                    
                    # تنظیمات و اجرا
                    CONFIG = {
                        "file_path": r'second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx',
                        "output_path": r'outputs\G11\dsas_g11_lubrication_system_clustering\clustering\dsas_g11_lubrication\clustering_g11_lubrication_output2.xlsx',
                        "sensors": ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']
                    }

                    # اجرا با DBSCAN
                    analyzer = SmartBearingAnalyzer(
                        CONFIG["file_path"], 
                        CONFIG["output_path"], 
                        CONFIG["sensors"],
                        eps=0.5,
                        min_samples=5
                    )
                    result = analyzer.run_analysis()
                    
                    if result is not None:
                        print("\n" + "="*60)
                        print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                        print("="*60)
                    else:
                        print("\n" + "="*60)
                        print("❌ اجرای زمان‌بندی شده با شکست مواجه شد!")
                        print("="*60)
                    
                    # ثبت زمان اجرا
                    last_run_time = current_time
                    
                    # 10 ثانیه صبر کن تا از اجرای مجدد در همان دقیقه جلوگیری شود
                    time.sleep(10)
            
            # هر 10 ثانیه یکبار بررسی کن
            time.sleep(10)
            
        except KeyboardInterrupt:
            print("\n" + "="*60)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*60)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            print("🔄 ادامه اجرا...")
            time.sleep(60)

# اجرای اصلی
if __name__ == "__main__":
    try:
        print("="*60)
        print("🚀 شروع برنامه تحلیل خوشه‌بندی سیستم روغن‌کاری با DBSCAN")
        print("="*60)
        
        # شروع زمان‌بندی
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        input("برای خروج Enter بزنید...")

🚀 شروع برنامه تحلیل خوشه‌بندی سیستم روغن‌کاری با DBSCAN
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - تحلیل خوشه‌بندی سیستم روغن‌کاری با DBSCAN
⏰ زمان‌های اجرا (هر روز):
   - ساعت 10:00
   - ساعت 10:05
   - ساعت 10:10
💡 برای توقف برنامه، Ctrl+C را بزنید

⏰ زمان اجرا فرا رسید: 2026-07-02 22:17:19
🔄 شروع تحلیل در 2026-07-02 22:17:19
✅ داده بارگذاری شد. تعداد رکوردها: 7,117
✅ پیش‌پردازش انجام شد. تعداد رکوردها: 7,113
🔄 مرحله 1: استانداردسازی داده‌ها...
🔄 مرحله 2: کلاسترینگ با DBSCAN (eps=0.5, min_samples=5)...
   تعداد خوشه‌ها: 35
   تعداد نقاط نویز: 266 (3.74%)
🔄 مرحله 3: محاسبه شاخص تخریب...
   محدوده شاخص تخریب: 0.0000 تا 5.0500
🔄 مرحله 4: لیبل‌گذاری وضعیت سلامت...

📊 توزیع وضعیت‌ها:
   Investigation Needed (Operational Drift): 4,344 (61.07%)
   Healthy (Optimal Performance): 2,264 (31.83%)
   Observation Required (Pattern Change): 505 (7.10%)
🔄 مرحله 5: فیلتر کردن داده‌های ۳۰ روز آخر...
   بازه خروجی: 2026-04-08 20:38:09 تا 2026-05-08 20:38:09
   تعداد رکوردهای ۳۰ روز آخر: 102

📊 آمار ن